# Семинар 11 [17-25-2]. Распределения в SciPy: документация и эксперименты

За 90 минут сравним шесть распределений в трёх лабораториях: проверка партии,
ошибки измерений и срок службы оборудования. Затем найдём ошибки в чужом коде
и самостоятельно решим новую задачу.

**Порядок работы:** документация → параметры и методы → ваш код → симуляция → вывод.
Перед расчётом прочитайте указанные части документации и заполните короткую заметку.

**Как устроены лаборатории.** Лаборатория 1 — разобранный образец: код уже написан,
нужно прочитать его, понять каждую строку и дописать два места с пометкой **TODO**.
В лаборатории 2 готова половина кода. Лабораторию 3 и задачу на перенос вы пишете
сами; под условием есть план и подсказки, которые можно раскрыть, если застряли.

Графики и базовую обработку выборок выполняют готовые функции. Вы задаёте законы,
считаете вероятности и перцентили, создаёте выборки и объясняете результаты.
Все ситуации — авторские учебные эксперименты с заданными моделями, без подбора параметров.

**Запуск:** в Colab загрузите скачанный `.ipynb` через «Файл → Загрузить блокнот»,
сохраните копию на Диске и выполняйте ячейки по порядку (`Shift+Enter`).
После заполнения перезапустите среду и выполните всё сверху вниз.
Для незаполненных блоков готовые функции сообщат, каких результатов пока не хватает.

Датасеты скачивать не нужно. Для чтения документации и работы в Colab нужен интернет;
локально вычисления работают с заранее установленными NumPy, pandas, Matplotlib и SciPy.

## Маршрут на 90 минут

| Время | Действие |
|---|---|
| 0–8 | Ожидание, дисперсия, квантиль и перцентиль |
| 8–13 | Как найти параметры и методы в документации |
| 13–25 | Лаборатория 1, образец: биномиальный и гипергеометрический законы |
| 25–43 | Лаборатория 2, половина кода: равномерный и нормальный законы |
| 43–65 | Лаборатория 3, самостоятельно: показательный закон и закон Вейбулла |
| 65–75 | Найдите ошибки |
| 75–87 | Самостоятельный перенос |
| 87–90 | Итоги и собственная шпаргалка |

Не успели — закончите дома: лаборатория 3 и перенос входят в домашнюю работу.

## 0. Подготовка и короткий рекап

$E(X)$ — характеристика модели, среднее выборки — характеристика конкретных наблюдений.
$\mathrm{Var}(X)=E(X^2)-E(X)^2$, $\sigma(X)=\sqrt{\mathrm{Var}(X)}$.
Дисперсия имеет квадрат единиц исходной величины, стандартное отклонение — исходные единицы.

**Знак неравенства.** В курсе $F(x)=P(X<x)$, а в SciPy `cdf(x)` означает $P(X\le x)$.
Для непрерывного закона разницы нет: вероятность отдельной точки нулевая.
Для целочисленной величины разница есть, и её легко перепутать:

| Событие | Вызов SciPy |
|---|---|
| $P(X\le k)$ | `cdf(k)` |
| $P(X<k)=F(k)$ | `cdf(k - 1)` |
| $P(X>k)$ | `sf(k)` |
| $P(X\ge k)$ | `sf(k - 1)` |

Перед вызовом метода запишите событие словами и проверьте, включена ли граница.

**Квантиль и перцентиль.** Квантиль уровня $q$ — порог, который величина не превышает
с вероятностью не меньше $q$. Метод `ppf(q)` возвращает наименьшее такое значение.
Для непрерывного закона в квантиле $P(X\le x_q)$ ровно равна $q$.
Для дискретного закона это наименьшее целое $k$, для которого `cdf(k)` $\ge q$:
равенства может не быть вовсе. **95-й перцентиль** — квантиль уровня $0{,}95$.
Обратите внимание на шкалу: в `ppf` уровень — доля (`0.95`), в `np.percentile` — проценты (`95`).

Спросите себя: достаточно ли среднего для оценки редкой большой ошибки?
Чем характеристика модели отличается от её оценки по 10 000 наблюдений?

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

rng = np.random.default_rng(2026)
SAMPLE_SIZE = 10_000
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

### Готовые функции

Выполните следующую ячейку один раз. Разбирать код рисования сейчас не требуется.

| Функция | Что ей передать | Что получите |
|---|---|---|
| `plot_laws` | Словарь «имя → созданный объект распределения» и настройки осей | PMF/PDF и CDF |
| `compare_samples` | Тот же словарь и словарь «имя → ваша выборка» | Выборки рядом с теоретическими кривыми |
| `summarize_samples` | Словарь выборок | Объём, среднее, дисперсию и стандартное отклонение выборки |
| `show_results` | Вашу таблицу: список словарей или DataFrame | Аккуратно оформленные результаты |

`summarize_samples` делит сумму квадратов отклонений на $n$, а не на $n-1$.
Теоретические характеристики, вероятности, условные частоты и перцентили вычисляете вы.
Имена моделей в словарях законов и выборок должны совпадать.

In [ ]:
# @title Готовые функции для графиков и выборок (ячейку можно свернуть)
COLORS = ("#1F6BFF", "#6B7280")
LINESTYLES = ("-", "--")


def _sample_values(sample):
    values = np.asarray(sample, dtype=float)
    if values.ndim != 1 or values.size == 0 or not np.isfinite(values).all():
        raise ValueError("Нужна непустая одномерная выборка конечных чисел.")
    return values


def _ready(mapping, description):
    if mapping is None or len(mapping) == 0:
        print(f"Сначала заполните расчётную ячейку: {description}.")
        return False
    return True


def show_results(rows):
    """Показывает вашу таблицу результатов; сама ничего не считает."""
    if not _ready(rows, "таблица ваших результатов"):
        return
    frame = pd.DataFrame(rows).round(5)
    if len(frame.columns) <= 6:
        display(frame)
    else:
        for start in range(1, len(frame.columns), 5):
            display(frame.iloc[:, [0, *range(start, min(start + 5, len(frame.columns)))]] )


def summarize_samples(samples):
    """Среднее, дисперсия (делим на n) и стандартное отклонение выборок."""
    if not _ready(samples, "словарь выборок"):
        return pd.DataFrame()
    rows = []
    for name, sample in samples.items():
        values = _sample_values(sample)
        rows.append({
            "Модель": name,
            "Объём выборки": values.size,
            "Среднее выборки": values.mean(),
            "Дисперсия выборки (делим на n)": values.var(ddof=0),
            "Стандартное отклонение выборки": values.std(ddof=0),
        })
    return pd.DataFrame(rows)


def _grid(xlim, discrete):
    lower, upper = xlim
    if discrete:
        return np.arange(np.floor(lower), np.ceil(upper) + 1, dtype=int)
    return np.linspace(lower, upper, 700)


def _decorate(axes, xlim, xlabel, density_label, discrete, thresholds):
    axes[0].set_title("Вероятности значений" if discrete else "Плотность")
    axes[0].set_ylabel("Вероятность" if discrete else density_label)
    axes[1].set_title("CDF: вероятность не превысить порог")
    axes[1].set_ylabel("Вероятность")
    axes[0].set_ylim(bottom=0)
    axes[1].set_ylim(-0.02, 1.02)
    for ax in axes:
        ax.set_xlim(*xlim)
        ax.set_xlabel(xlabel)
        for threshold in thresholds:
            ax.axvline(threshold, color="#374151", ls=":", lw=1.2,
                       label=f"Порог {threshold:g}")
        ax.legend(fontsize=9)


def plot_laws(laws, *, discrete, xlim, xlabel, density_label="Плотность", thresholds=()):
    """Рисует PMF/PDF и CDF переданных объектов распределений."""
    if not _ready(laws, "словарь объектов распределений"):
        return
    grid = _grid(xlim, discrete)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
    width = 0.8 / len(laws)
    for index, (name, law) in enumerate(laws.items()):
        color = COLORS[index % len(COLORS)]
        linestyle = LINESTYLES[index % len(LINESTYLES)]
        if discrete:
            offset = (index - (len(laws) - 1) / 2) * width
            axes[0].bar(grid + offset, law.pmf(grid), width=width,
                        color=color, alpha=0.75, label=name)
            axes[1].step(grid, law.cdf(grid), where="post", color=color,
                         ls=linestyle, lw=2, label=name)
        else:
            axes[0].plot(grid, law.pdf(grid), color=color, ls=linestyle,
                         lw=2, label=name)
            axes[1].plot(grid, law.cdf(grid), color=color, ls=linestyle,
                         lw=2, label=name)
    _decorate(axes, xlim, xlabel, density_label, discrete, thresholds)
    plt.show()


def compare_samples(laws, samples, *, discrete, xlim, xlabel,
                    density_label="Плотность", thresholds=()):
    """Сравнивает выборки с теоретическими PMF/PDF и CDF (эмпирическая CDF — P(X ≤ x))."""
    if not _ready(laws, "словарь объектов распределений"):
        return
    if not _ready(samples, "словарь выборок"):
        return
    if set(laws) != set(samples):
        raise ValueError("Имена моделей в словарях распределений и выборок должны совпадать.")
    grid = _grid(xlim, discrete)
    for index, (name, law) in enumerate(laws.items()):
        values = _sample_values(samples[name])
        color = COLORS[index % len(COLORS)]
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
        fig.suptitle(f"{name}: {values.size:,} наблюдений".replace(",", " "))
        if discrete:
            support, counts = np.unique(values, return_counts=True)
            axes[0].bar(support, counts / values.size, width=0.75,
                        color=color, alpha=0.4, label="Частоты в выборке")
            axes[0].plot(grid, law.pmf(grid), "o", ms=4,
                         color="#111827", label="Теоретические вероятности")
            ecdf_grid = grid
        else:
            axes[0].hist(values, bins=40, density=True, color=color,
                         alpha=0.4, label="Гистограмма выборки")
            axes[0].plot(grid, law.pdf(grid), color="#111827", lw=2,
                         label="Теоретическая плотность")
            visible = values[(values >= xlim[0]) & (values <= xlim[1])]
            ecdf_grid = np.unique(np.r_[xlim[0], visible, xlim[1]])
        ecdf_values = np.searchsorted(np.sort(values), ecdf_grid, side="right") / values.size
        axes[1].step(ecdf_grid, ecdf_values, where="post", color=color,
                     lw=1.7, label="Эмпирическая CDF")
        if discrete:
            axes[1].step(grid, law.cdf(grid), where="post", color="#111827",
                         ls="--", lw=1.5, label="Теоретическая CDF")
        else:
            axes[1].plot(grid, law.cdf(grid), color="#111827", ls="--",
                         lw=1.5, label="Теоретическая CDF")
        _decorate(axes, xlim, xlabel, density_label, discrete, thresholds)
        plt.show()

### Учимся находить нужную операцию

Откройте [Probability distributions](https://docs.scipy.org/doc/scipy/tutorial/stats/probability_distributions.html): **Common methods**,
**Shifting and scaling**, **Freezing a distribution**. Найдите, как создать объект с
фиксированными параметрами и затем вызвать метод этого объекта.

Небольшая готовая демонстрация ниже использует стандартный нормальный закон.
Найдите в документации смысл двух методов и предскажите результат до запуска.
Для своих лабораторий сначала прочитайте описание параметров конкретного закона.

In [ ]:
demo_law = stats.norm()
show_results([{
    "Пример": "Стандартный нормальный закон",
    "CDF в нуле": demo_law.cdf(0),
    "Квантиль уровня 0.5": demo_law.ppf(0.5),
}])

## 1. Проверка партии: независимость и выборка без возвращения

**Сначала документация, около 3 минут:** [binom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html),
[hypergeom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html). Нужны **Methods**, **Notes**, **Examples**.

**Условие.** В первой схеме независимо проверяют 10 изделий; каждое бракованное
с вероятностью 0,2. Во второй из партии 20 изделий, среди которых ровно 4
бракованных, случайно выбирают 10 без возвращения. $X$ — число бракованных
среди проверенных. Каждая повторная проверка второй схемы начинается с исходной партии.

Перед кодом выясните: какой закон относится к каждой схеме; что означают `n`, `p`,
`M`, `N` на страницах; какие методы дают характеристики, вероятность события,
квантиль и случайную выборку. Не переносите буквы из условия в аргументы механически:
у `hypergeom` буква `n` означает совсем не то, что у `binom`.

**Заметка перед кодом — заполните своими словами.**

- Величина и возможные значения: …
- Первая схема: семейство, параметры и их смысл: …
- Вторая схема: семейство, параметры и их смысл: …
- Нужные методы и смысл их результатов: …

### 1.1. Образец анализа моделей

Ниже готовое решение. Прочитайте его построчно и сверьте с документацией:
почему у `hypergeom` именно такие аргументы, почему $P(X\ge3)$ считается как `sf(2)`,
зачем выводятся `cdf` в перцентиле и в предыдущем целом.

**TODO 1.** Допишите $P(X<2)$ — вероятность, что бракованных меньше двух.
По определению курса это $F(2)$. Посмотрите таблицу знаков в рекапе: какой вызов `cdf` нужен?

In [ ]:
# Образец. Так оформляем расчёт в каждой лаборатории.
# 1) Закон и его параметры. Имена ключей потом используются в графиках и выборках.
batch_laws = {
    "С возвращением (binom)": stats.binom(n=10, p=0.2),
    # В hypergeom: M — вся партия, n — бракованные в партии, N — сколько изделий проверяем
    "Без возвращения (hypergeom)": stats.hypergeom(M=20, n=4, N=10),
}

# 2) Для каждой модели считаем нужные величины и складываем их в строку таблицы.
batch_results = []
for name, law in batch_laws.items():
    q95 = law.ppf(0.95)  # 95-й перцентиль: наименьшее k, для которого cdf(k) >= 0,95
    batch_results.append({
        "Модель": name,
        "E(X)": law.mean(),
        "Var(X)": law.var(),
        "σ(X)": law.std(),
        "P(X ≥ 3)": law.sf(2),  # «хотя бы три» = «больше двух»; sf(2) = P(X > 2)
        "P(X < 2)": None,  # TODO 1: F(2) = P(X < 2). Какой вызов cdf нужен?
        "95-й перцентиль": q95,
        "cdf в перцентиле": law.cdf(q95),          # уже не меньше 0,95
        "cdf на шаг левее": law.cdf(q95 - 1),      # ещё меньше 0,95, значит порог наименьший
    })

In [ ]:
show_results(globals().get("batch_results"))
plot_laws(globals().get("batch_laws"), discrete=True, xlim=(-0.5, 10.5),
          xlabel="Число бракованных изделий, шт.", thresholds=(3,))

### 1.2. Эксперимент

Генерация уже написана: у каждого объекта есть метод `rvs`, а `random_state=rng`
делает результат воспроизводимым. Проверьте в [hypergeom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html),
что выборка — это числа бракованных, а не список извлечённых изделий.

**TODO 2.** Посчитайте частоту события $X\ge3$ в выборке: долю наблюдений, где `sample >= 3`.
Подсказка: среднее от массива `True/False` и есть доля `True`.

Для дискретной выборки перцентиль считаем с `method="inverted_cdf"`: так он определяется
тем же правилом «наименьшее значение, где доля не меньше уровня», что и `ppf`.

In [ ]:
batch_samples = {
    name: law.rvs(size=SAMPLE_SIZE, random_state=rng)
    for name, law in batch_laws.items()
}

batch_experiment = []
for name, sample in batch_samples.items():
    law = batch_laws[name]
    batch_experiment.append({
        "Модель": name,
        "P(X ≥ 3), модель": law.sf(2),
        "Частота X ≥ 3": None,  # TODO 2: доля наблюдений, где sample >= 3
        "95-й перцентиль, модель": law.ppf(0.95),
        "95-й перцентиль, выборка": np.percentile(sample, 95, method="inverted_cdf"),
    })

In [ ]:
show_results(globals().get("batch_experiment"))
show_results(summarize_samples(globals().get("batch_samples")))
compare_samples(globals().get("batch_laws"), globals().get("batch_samples"),
                discrete=True, xlim=(-0.5, 10.5),
                xlabel="Число бракованных изделий, шт.", thresholds=(3,))

### 1.3. Ваш вывод

Почему ожидания совпали, а дисперсии различаются? Как выборка без возвращения
меняет неопределённость? Почему CDF в 95-м перцентиле не обязана быть ровно 0,95?
Объясните небольшое различие между частотой и вероятностью.
Почему у гипергеометрического закона $P(X<2)$ совпала с $P(X\ge3)$?

**Запишите 2–4 предложения:** …

## 2. Ошибки измерения: одинаковые характеристики, разные хвосты

**Сначала документация, около 3 минут:** [uniform](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.uniform.html),
[norm](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html), [numpy.percentile](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html).

**Условие.** Ошибка первого прибора нормальна с ожиданием 0 мм и стандартным
отклонением 2 мм. Ошибка второго равномерна на $[-2\sqrt3;2\sqrt3]$ мм.
Нас интересует выход ошибки за пределы $\pm4$ мм.

Найдите в **Notes** и **Examples**, как границы отрезка связаны с `loc`, `scale`,
что означает `scale` у нормального закона, чем плотность отличается от вероятности.
В **Methods** выберите операции для характеристик, двух хвостов и квантилей.
В документации NumPy проверьте шкалу уровня перцентиля и параметр `method`.

**Заметка перед кодом — заполните.**

- Величина, единицы, возможные значения для двух приборов: …
- Параметры SciPy и их связь с условием: …
- Методы для вероятности выхода за две границы и квантилей: …
- Чем различаются аргументы уровня в `ppf` и `numpy.percentile`: …

### 2.1. Анализ моделей: половина кода готова

Используйте [uniform](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.uniform.html) и [norm](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html).

Нормальный закон и характеристики уже посчитаны. Допишите места с пометкой **TODO**:

- **TODO 3.** Равномерный закон на $[-2\sqrt3;\,2\sqrt3]$. В **Notes** страницы `uniform`
  найдите, как отрезок $[a;b]$ записывается через `loc` и `scale`. Корень — `np.sqrt`.
- **TODO 4.** $P(|X|>4)$. Это два хвоста: $P(X<-4)+P(X>4)$. Какой метод даёт вероятность
  слева от порога, а какой — справа?
- **TODO 5.** 95-й и 99-й перцентили модели.

Когда всё заполните, проверьте, совпадают ли ожидания и дисперсии двух приборов.
Положительный 95-й перцентиль и симметричный интервал, содержащий 95% ошибок,
отвечают на разные вопросы. Здесь сначала требуется именно перцентиль.

In [ ]:
error_laws = {
    "Нормальный": stats.norm(loc=0, scale=2),  # scale — стандартное отклонение, а не дисперсия
    "Равномерный": None,  # TODO 3: отрезок [−2√3; 2√3]; как связаны loc, scale и границы a, b?
}

error_results = []
for name, law in error_laws.items():
    if law is None:
        print(f"Модель «{name}» ещё не задана.")
        continue
    error_results.append({
        "Модель": name,
        "E(X), мм": law.mean(),
        "Var(X), мм²": law.var(),
        "σ(X), мм": law.std(),
        "P(|X| > 4)": None,  # TODO 4: левый хвост + правый хвост
        "95-й перцентиль, мм": None,  # TODO 5
        "99-й перцентиль, мм": None,  # TODO 5
    })

In [ ]:
show_results(globals().get("error_results"))
plot_laws({k: v for k, v in globals().get("error_laws", {}).items() if v is not None},
          discrete=False, xlim=(-7, 7),
          xlabel="Ошибка измерения, мм", density_label="Плотность, 1/мм",
          thresholds=(-4, 4))

### 2.2. Эксперимент

Генерация снова готова. Для перцентилей выборки откройте [numpy.percentile](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html):
какой шкалы уровень он принимает? Для непрерывных выборок используйте `method="linear"`.

**TODO 6.** Частота $|X|>4$ в выборке (модуль — `np.abs`).
**TODO 7.** 95-й и 99-й перцентили выборки.

Сравните результаты с модельными значениями из таблицы 2.1.
Характеристики выборок отдельно выведет готовая функция.

In [ ]:
error_samples = {
    name: law.rvs(size=SAMPLE_SIZE, random_state=rng)
    for name, law in error_laws.items() if law is not None
}

error_experiment = []
for name, sample in error_samples.items():
    error_experiment.append({
        "Модель": name,
        "Частота |X| > 4": None,  # TODO 6
        "95-й перцентиль выборки, мм": None,  # TODO 7
        "99-й перцентиль выборки, мм": None,  # TODO 7
    })

In [ ]:
show_results(globals().get("error_experiment"))
show_results(summarize_samples(globals().get("error_samples")))
compare_samples({k: v for k, v in globals().get("error_laws", {}).items() if v is not None},
                globals().get("error_samples"),
                discrete=False, xlim=(-7, 7), xlabel="Ошибка измерения, мм",
                density_label="Плотность, 1/мм", thresholds=(-4, 4))

### 2.3. Ваш вывод

Достаточно ли одинаковых ожидания и дисперсии для одинакового риска больших ошибок?
Почему в одной выборке выходов за допуск может не быть вовсе?
Чем модельный перцентиль отличается от выборочного?

**Запишите 2–4 предложения:** …

## 3. Срок службы: новое и уже работавшее устройство

**Сначала документация, около 3 минут:** [expon](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.expon.html),
[weibull_min](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.weibull_min.html).

**Условие.** У первого типа устройств $T$ имеет показательное распределение
с интенсивностью 0,01 в час. У второго — распределение Вейбулла с формой 2
и масштабом 100 часов. Сдвиг обеих моделей нулевой. Параметры заданы,
оценивать их по наблюдениям не нужно.

В **Notes** найдите связь интенсивности со `scale`, смысл формы Вейбулла
и масштаба. В **Methods** найдите вероятность проработать дольше порога,
характеристики и квантили. В **Examples** проверьте порядок аргументов.
Подумайте, какой уровень нужен для срока, к которому откажет лишь 10% устройств.

**Заметка перед кодом — заполните.**

- Величина, единицы и область возможных значений: …
- Параметры SciPy для двух моделей: …
- Метод для вероятности проработать дольше порога: …
- Уровень квантиля для замены до отказа 10% устройств и его смысл: …

### 3.1. Ваш анализ моделей — самостоятельно

Вернитесь к [expon](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.expon.html) и [weibull_min](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.weibull_min.html).

Создайте словарь `lifetime_laws`. Для каждого закона вычислите:

- $E(T)$, $\mathrm{Var}(T)$ и $\sigma(T)$;
- $P(T\le50)$ и $P(T>100)$;
- 10-й и 90-й перцентили;
- срок замены, к которому откажет не более 10% устройств, и вероятность отказа к нему;
- $P(T>150\mid T>50)$ — вероятность ещё 100 часов работы после уже отработанных 50.

Сравните последний результат с $P(T>100)$ для нового устройства. Выведите таблицу `lifetime_results`
— список словарей, как в лабораториях 1 и 2. Время измеряется в часах, дисперсия — в часах².

**План.** Повторите структуру лаборатории 2: словарь законов → цикл по нему → словарь-строка таблицы.
Каждая строка расчёта — один вызов метода.

<details>
<summary>Подсказка 1: параметры</summary>

В **Notes** страницы `expon` плотность записана с параметром `scale`. Сравните её с
$\lambda e^{-\lambda x}$ из лекции: чему равен `scale` при $\lambda=0{,}01$ в час?
У `weibull_min` параметр формы называется `c`, масштаб — `scale`.
</details>

<details>
<summary>Подсказка 2: срок замены</summary>

«К сроку откажет 10% устройств» означает $P(T\le t)=0{,}1$. Какой это перцентиль?
Проверьте ответ: `cdf` в найденном сроке должна вернуть 0,1.
</details>

<details>
<summary>Подсказка 3: условная вероятность</summary>

$P(A\mid B)=\dfrac{P(AB)}{P(B)}$. Если устройство проработало больше 150 часов,
оно тем более проработало больше 50, поэтому $AB=A$ и
$P(T>150\mid T>50)=\dfrac{P(T>150)}{P(T>50)}$. Какой метод даёт каждую из вероятностей?
</details>

In [ ]:
# Ваш код: lifetime_laws, цикл по моделям, lifetime_results.


In [ ]:
show_results(globals().get("lifetime_results"))
plot_laws(globals().get("lifetime_laws"), discrete=False, xlim=(0, 500),
          xlabel="Срок службы, ч", density_label="Плотность, 1/ч",
          thresholds=(50, 100))

### 3.2. Ваш эксперимент — самостоятельно

Сгенерируйте по `SAMPLE_SIZE` сроков службы с `random_state=rng` в словарь `lifetime_samples`.
Посчитайте 10-й и 90-й перцентили выборки (`method="linear"`) и частоты трёх событий:
отказ к 50 часам, работа дольше 100 часов, отказ к **модельному** сроку замены.

Отдельно оцените условную вероятность: сначала оставьте только устройства,
дожившие до 50 часов, затем среди них найдите долю доживших до 150.
Укажите размер этой подвыборки. Сравнивать её нужно с условной вероятностью модели.
Сохраните таблицы `lifetime_experiment` и `lifetime_conditional_experiment`.

<details>
<summary>Подсказка: как отобрать подвыборку</summary>

Маска `sample > 50` — массив `True/False`. Запись `sample[sample > 50]` оставляет только
те наблюдения, где стоит `True`. Её длина `len(...)` — знаменатель условной частоты.
</details>

In [ ]:
# Ваш код: lifetime_samples, lifetime_experiment, lifetime_conditional_experiment.


In [ ]:
show_results(globals().get("lifetime_experiment"))
show_results(globals().get("lifetime_conditional_experiment"))
show_results(summarize_samples(globals().get("lifetime_samples")))
compare_samples(globals().get("lifetime_laws"), globals().get("lifetime_samples"),
                discrete=False, xlim=(0, 500), xlabel="Срок службы, ч",
                density_label="Плотность, 1/ч", thresholds=(50, 100))

### 3.3. Ваш вывод

Что означает 10-й перцентиль срока службы? Что случится, если заменить его 90-м?
Как уже отработанные 50 часов меняют вероятность следующих 100 часов работы?
Почему при оценке условной частоты меняется знаменатель?

**Запишите 2–4 предложения:** …

## 4. Найдите ошибки

В каждой строке ниже есть ошибка: код запускается, но отвечает не на тот вопрос.
Для каждой строки найдите ошибку, исправьте вызов и в комментарии одной фразой объясните, что было не так.
Сначала предскажите правильный ответ, затем запустите.

1. $X\sim N$ с $E(X)=40$ и $\mathrm{Var}(X)=100$. Нужна дисперсия, чтобы проверить параметры.
2. Диаметр детали $X\sim N(20;\,0{,}1^2)$, мм. Нужна $P(X=20)$.
3. $N$ — число заявок за час, закон Пуассона с $\lambda=3$. Нужна $F(2)=P(N<2)$.
4. $X\sim N(0;1)$. Нужен 95-й перцентиль.
5. Заявки приходят с интенсивностью 3 в час. Нужно среднее время ожидания заявки в **минутах**.
6. $X$ равномерна на отрезке $[2;7]$ (задача 6.2 методички). Нужно $E(X)$.

In [ ]:
count = stats.poisson(mu=3)  # число заявок за час

print("1. Var(X):", stats.norm(loc=40, scale=100).var())
print("2. P(X = 20):", stats.norm(loc=20, scale=0.1).pdf(20))
print("3. P(N < 2):", count.cdf(2))
print("4. 95-й перцентиль:", stats.norm().ppf(95))
print("5. E(T), мин:", stats.expon(scale=3).mean())
print("6. E(X):", stats.uniform(loc=2, scale=7).mean())

**Ваши объяснения (по одной фразе на строку):** 1 — …; 2 — …; 3 — …; 4 — …; 5 — …; 6 — …

## 5. Самостоятельный перенос: допуск на размер

**Условие.** Размер изделия $X$ распределён нормально: $E(X)=100$ мм,
$\mathrm{Var}(X)=3^2$ мм². Допустимы размеры от 94 до 106 мм включительно.

1. Найдите вероятность выхода за допуск.
2. Постройте симметричный относительно ожидания интервал, содержащий 99% значений.
   Объясните, как вероятность распределяется между хвостами.
3. Проверьте один из результатов симуляцией и объясните расхождение с теорией.

Используйте [документацию нормального распределения](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
самостоятельно. Здесь нет списка шагов по методам и обязательных имён переменных.
Для вывода своей таблицы можно вызвать `show_results`.

In [ ]:
# Решите задачу самостоятельно по документации.
# Запишите оба события, посчитайте и проверьте один результат симуляцией.

**Ваш ответ:** вероятность выхода за допуск — …; интервал — …;
вероятности в левом и правом хвостах — …; результат симуляции и его смысл — …

## 6. Собственная шпаргалка и самопроверка

Заполните таблицу после выполнения лабораторий. Смысл метода должен быть понятен
без кода: для вероятности обязательно укажите знак неравенства.

| Вопрос | Метод | Смысл аргументов | Что возвращается | Ссылка на определение |
|---|---|---|---|---|
| Ожидание, дисперсия, стандартное отклонение | … | … | … | … |
| Вероятность отдельного значения / плотность | … | … | … | … |
| Вероятность слева от порога, $\le$ и $<$ для целочисленной величины | … | … | … | … |
| Вероятность справа от порога | … | … | … | … |
| Квантиль модели | … | … | … | … |
| Случайная выборка | … | … | … | … |
| Перцентиль наблюдений | … | … | … | … |

Сверьтесь с [Common methods](https://docs.scipy.org/doc/scipy/tutorial/stats/probability_distributions.html#common-methods),
[определением дискретного квантиля](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.ppf.html)
и [numpy.percentile](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html).

Перед завершением проверьте:

- Для каждой лаборатории есть параметры, расчётная таблица, графики, симуляция и вывод.
- Вероятности лежат в $[0;1]$, дисперсии неотрицательны, единицы подписаны.
- Для дискретного события явно учтено включение границы.
- 95-й перцентиль модели и симметричный 95%-й интервал не перепутаны.
- Условная частота использует правильную подвыборку и указанный знаменатель.
- Перезапуск среды и выполнение сверху вниз воспроизводят ваш анализ.

Если остались незавершённые расчёты, закончите их дома. По желанию повторите одну
симуляцию с другим зерном: теоретические значения сохранятся, выборочные немного изменятся.
Не подбирайте зерно ради точного совпадения. Совпадение графиков симуляции и модели
здесь проверяет работу с заданным законом, а не доказывает пригодность закона реальным данным.